# 🕵️ كشف الاحتيال في المعاملات (Fraud Detection — Anomaly + Imbalanced ML)
### مشروع B4 — مسار تعلم الآلة (Machine Learning Track)

---
## 🎯 المشكلة التجارية (Business Problem)
شركة مدفوعات عايزة **تكتشف المعاملات الاحتيالية** لحظياً. التحدي: الاحتيال **نادر جداً** (أقل من 4%)،
وفريق المراجعة يقدر يراجع **عدد محدود** من المعاملات يومياً — فلازم نرتّبهم بالأخطر.

**نوع المشكلة:** تصنيف ثنائي **غير متوازن بشدة** + كشف شذوذ (Anomaly Detection).

## 📦 ما الذي يثبته المشروع (يختلف عن مخاطر الائتمان)
**كشف الشذوذ غير الموجّه (Isolation Forest)** · التعامل مع عدم التوازن الشديد ·
**PR-AUC** بدل ROC-AUC · **الدقة عند حدّ المراجعة (Precision@k)** · مقارنة موجّه vs غير موجّه.


## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

In [ ]:
# 🚀 إعداد تلقائي — Colab / Kaggle / محلي (no-op محلياً)
import os, sys, subprocess, importlib.util
REPO    = "https://github.com/Ahmedelgendyyy/data-science-portfolio"
PROJECT = "ml/b4_fraud_detection"          # مسار المشروع داخل portfolio/
PKGS    = ["xgboost"]          # مكتبات المشروع (تتثبّت لو ناقصة)
for _pkg in PKGS:
    if importlib.util.find_spec(_pkg.replace("-", "_")) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg])
if not os.path.isdir("data"):                 # سحابياً: نجيب الريبو ونروح لمجلد المشروع
    _clone = REPO.rstrip("/").split("/")[-1]
    if not os.path.isdir(_clone):
        subprocess.run(["git", "clone", "-q", REPO + ".git"])
    os.chdir(os.path.join(_clone, "portfolio", PROJECT))
print("✓ جاهز —", os.getcwd())

## 📚 قبل ما تبدأ — محتاج تذاكر إيه
| المفهوم | المصدر | بيُستخدم في إيه |
|---|---|---|
| عدم التوازن الشديد (Rare events) | Géron — *Hands-On ML* (ch.3) | الاحتيال < 4% — accuracy عديمة الفائدة |
| **كشف الشذوذ (Isolation Forest)** | Géron (ch.9) / Liu et al. | اكتشاف الغريب **بدون labels** |
| **PR-AUC vs ROC-AUC** | Davis & Goadrich | PR-AUC أصدق مع الأحداث النادرة |
| **Precision@k** | أدبيات استرجاع المعلومات | "لو راجعنا أعلى 100، كام منهم احتيال؟" — مقياس عملي |
| scale_pos_weight / class_weight | Thakur / Géron | موازنة الموديل الموجّه |

> 🎯 **بيُستخدم في الواقع:** البنوك، بطاقات الائتمان، التأمين، كشف الحسابات الوهمية، أمن الشبكات.


## 0️⃣ المكتبات

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_style('whitegrid'); np.random.seed(42)
print('ready ✓')

## 1️⃣ تحميل واستكشاف البيانات (EDA)

In [ ]:
df = pd.read_csv('data/fraud_data.csv')
# TODO: اطبع shape ومعدل الاحتيال، وارسم العلاقات
print(df['is_fraud'].mean())

> 📌 الاحتيال نادر جداً → موديل يقول "كله سليم" هيحقّق دقة 96%+ لكنه **عديم الفائدة**. لذلك نستخدم PR-AUC و Precision@k.

## 2️⃣ المعالجة والتقسيم

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
# TODO: افصل X/y، split stratified، ColumnTransformer (scale + onehot)، حوّل البيانات
...

## 3️⃣ كشف الشذوذ غير الموجّه (Isolation Forest) 🌲
**بدون استخدام الـ labels** — الموديل بيتعلّم "الطبيعي" ويعزل الغريب. نقيّم بعدين مقابل الحقيقة.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, roc_auc_score
# TODO: درّب IsolationForest بدون y، احسب anomaly_score = -score_samples، وقيّمه بـ PR-AUC
iso = ...
anomaly_score = ...

## 4️⃣ النماذج الموجّهة (Supervised) — مقارنة بالـ PR-AUC

In [ ]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
# TODO: احسب scale_pos_weight، درّب LogReg + XGBoost، قارن بالـ PR-AUC
...

## 5️⃣ الدقة عند حدّ المراجعة (Precision@k) 💼
فريق المراجعة يقدر يراجع **أعلى k معاملة مشبوهة يومياً**. السؤال العملي: **كام منهم احتيال فعلاً؟**

In [ ]:
best = results['XGBoost']
# TODO: رتّب المعاملات بالاحتمال تنازلياً، واحسب Precision@k و recall لأعلى k
...

## 6️⃣ منحنى Precision-Recall (المقياس الصح للأحداث النادرة)

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay
fig, ax = plt.subplots(figsize=(7,5))
PrecisionRecallDisplay.from_predictions(y_te, results['XGBoost'], name='XGBoost', ax=ax)
PrecisionRecallDisplay.from_predictions(y_te, results['LogReg'], name='LogReg', ax=ax)
PrecisionRecallDisplay.from_predictions(y_te, anomaly_score, name='IsolationForest', ax=ax)
ax.set_title('Precision-Recall Curves'); plt.show()

## 7️⃣ الخلاصة والتوصيات (Conclusion)
- **الموجّه أفضل من غير الموجّه:** XGBoost حقّق أعلى PR-AUC، لكن Isolation Forest مفيد لما **مفيش labels**.
- **Precision@k:** مراجعة أعلى ~100 معاملة بتمسك نسبة كبيرة من الاحتيال بأقل جهد — ده اللي يهمّ العمليات.
- **PR-AUC مش ROC-AUC:** مع 4% احتيال، ROC-AUC بيبالغ في التفاؤل؛ PR-AUC أصدق.
- **التوصية:** نظام هجين — XGBoost للترتيب اليومي + Isolation Forest لرصد أنماط جديدة، مع حدّ مراجعة حسب طاقة الفريق.
- **الخطوة القادمة:** ميزات زمنية (سرعة المعاملات)، وتحديث الموديل باستمرار لأن أنماط الاحتيال بتتغيّر.

> ✅ **اللي اتعلمته:** عدم التوازن الشديد، Isolation Forest، PR-AUC، و Precision@k.
